# Plot a patient's glucose data

Run each cell in order with Shift+Enter.

In [ ]:
from jupyterhealth_client import Code, JupyterHealthClient

client = JupyterHealthClient()

for study in client.list_studies():
    print(study["id"], study["name"])

Replace `<STUDY_ID>` with a study ID from the list above.

In [ ]:
study_id = <STUDY_ID>

for patient in client.list_patients(study_id=study_id):
    print(patient["id"], patient["nameGiven"], patient["nameFamily"])

Replace `<PATIENT_ID>` with a patient ID from the list above.

In [ ]:
patient_id = <PATIENT_ID>

df = client.list_observations_df(
    patient_id=patient_id, study_id=study_id, code=Code.BLOOD_GLUCOSE
)
df[["effective_time_frame_date_time_local", "blood_glucose_value"]].head()

Plot a typical day for this patient.
The line is the median glucose at each hour, the blue bands show how much it varies, and the green band is the target range.

In [ ]:
import matplotlib.pyplot as plt

hour = df["effective_time_frame_date_time_local"].dt.hour
glucose = df.groupby(hour)["blood_glucose_value"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhspan(70, 180, color="green", alpha=0.1, label="Target range")
ax.fill_between(glucose.quantile(0.05).index, glucose.quantile(0.05), glucose.quantile(0.95), color="tab:blue", alpha=0.2, label="5th-95th percentile")
ax.fill_between(glucose.quantile(0.25).index, glucose.quantile(0.25), glucose.quantile(0.75), color="tab:blue", alpha=0.4, label="25th-75th percentile")
ax.plot(glucose.median(), color="black", label="Median")
ax.set(xlabel="Hour of day", ylabel="Glucose (mg/dL)", xticks=range(0, 24, 3))
ax.legend(loc="upper left");